# SafeKV Figure 6

This notebook regenerates the cumulative-exposure figure used in the main evaluation. All plotted values are embedded below; it does not read experiment result files.

Output: `user_scripts/figures/cumulative_ledger_churn.pdf`

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

cwd = Path.cwd()
if cwd.name == "user_scripts":
    repo_root = cwd.parent
elif (cwd / "user_scripts").is_dir():
    repo_root = cwd
else:
    raise RuntimeError("Run this notebook from the repository root or user_scripts/")

figure_dir = repo_root / "user_scripts" / "figures"
figure_dir.mkdir(parents=True, exist_ok=True)

plt.rcParams.update(
    {
        "font.family": "serif",
        "font.size": 9,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    }
)

figure_dir

In [ ]:
# Figure 6: cumulative exposure across cache residencies.
# Embedded values: B=10 and 20 eviction/reinsertion cycles.
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

budget = 10
cycles = np.arange(1, 21)
strict_hits = np.zeros(20, dtype=int)
balanced_durable_hits = np.full(20, budget, dtype=int)
per_residency_hits = budget * cycles

series = (
    (strict_hits, "Strict ($B=0$)", "#377EB8", "o"),
    (balanced_durable_hits, "Balanced (durable)", "#009E73", "s"),
    (per_residency_hits, "Per-residency baseline", "#D55E00", "^"),
)

fig, ax = plt.subplots(figsize=(3.35, 2.45))
for values, label, color, marker in series:
    ax.plot(
        cycles,
        values,
        label=label,
        color=color,
        marker=marker,
        markevery=[0, 4, 9, 14, 19],
        markersize=3.2,
        linewidth=1.4,
    )

ax.axhline(budget, color="#4a5568", linestyle="--", linewidth=0.8)
ax.annotate("$20B=200$", (20, 200), xytext=(-4, 4), textcoords="offset points",
            ha="right", va="bottom", fontsize=7)
ax.set_xlabel("Eviction--reinsertion cycle")
ax.set_ylabel("Cumulative cross-principal hits")
ax.set_xlim(1, 20.5)
ax.set_xticks([1, 5, 10, 15, 20])
ax.set_ylim(0, 210)
ax.grid(axis="y", color="#D9D9D9", linewidth=0.6)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(loc="upper left", frameon=False, fontsize=6.3, handlelength=1.6)

# The inset makes the two contract modes visible despite the 200-hit baseline.
zoom = inset_axes(ax, width="43%", height="34%", loc="lower right", borderpad=1.0)
zoom.plot(cycles, strict_hits, color="#377EB8", linewidth=1.2)
zoom.plot(cycles, balanced_durable_hits, color="#009E73", linewidth=1.2)
zoom.axhline(budget, color="#4a5568", linestyle="--", linewidth=0.7)
zoom.set_xlim(1, 20)
zoom.set_ylim(-0.5, 12)
zoom.set_xticks([1, 20])
zoom.set_yticks([0, 10])
zoom.set_title("Contract modes", fontsize=6.5, pad=1)
zoom.tick_params(labelsize=6, pad=1)
zoom.grid(axis="y", color="#E5E5E5", linewidth=0.4)

fig.subplots_adjust(left=0.17, right=0.98, bottom=0.20, top=0.98)
fig.savefig(figure_dir / "cumulative_ledger_churn.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# The former AUC bar panel duplicated the primary end-to-end membership table
# and has been removed from Figure 6. The table remains authoritative because
# it reports AUC together with TPR, FPR, attack advantage, confidence intervals,
# and TTFT separation.